# 06. Clasificación asistida con RoBERTuito

Se utiliza el modelo preentrenado `pysentimiento/robertuito-sentiment-analysis` para **proponer** polaridad positiva, negativa o neutral en las publicaciones que no pertenecen a la muestra de prueba.

La propuesta automática no se presenta como etiqueta humana. Las columnas `etiqueta_humana` y `observacion_humana` quedan vacías para revisión posterior.


## A. Instalar las bibliotecas

- **Qué hace:** instala versiones compatibles de las bibliotecas del modelo.
- **Para qué:** disponer del tokenizador, los pesos de RoBERTuito y su preprocesador oficial.
- **Por qué:** fijar versiones hace que otra ejecución pueda reconstruir el mismo procedimiento.

Ejecuta esta celda una sola vez por sesión. Si Colab solicita reinicio, reinicia y continúa desde aquí.


In [ ]:
%pip install -q "transformers==4.49.0" "pysentimiento==0.7.3" "emoji==2.16.0"


## B. Cargar los datos y excluir la prueba

- **Qué hace:** lee el dataset del 05 y retira temporalmente los 100 `id` reservados.
- **Para qué:** impedir que la muestra independiente reciba sugerencias antes de la anotación humana.
- **Por qué:** ver la predicción podría sesgar la etiqueta de referencia.


In [ ]:
from pathlib import Path

if Path("/content").is_dir():
    from google.colab import drive
    drive.mount("/content/drive")
    RAIZ = Path("/content/drive/MyDrive/TESIS_DATA_SCIENCE/COLAB")
else:
    RAIZ = Path.cwd()

if not (RAIZ / "TABLAS").is_dir():
    raise FileNotFoundError("Abre este notebook desde la carpeta COLAB o ajusta RAIZ.")
print("Carpeta de trabajo:", RAIZ)

import json
import pandas as pd
import torch
from IPython.display import display
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.utils import get_column_letter
from pysentimiento.preprocessing import preprocess_tweet
from transformers import AutoModelForSequenceClassification, AutoTokenizer

DATASET = RAIZ / "TABLAS/05_preparacion/dataset_preparado_v4.xlsx"
PRUEBA = RAIZ / "TABLAS/05_preparacion/muestra_prueba_100_v4.xlsx"
datos = pd.read_excel(DATASET, sheet_name="datos", keep_default_na=False)
muestra = pd.read_excel(PRUEBA, sheet_name="evaluacion", usecols=["id"], keep_default_na=False)

esperadas = ["id", "fuente", "texto_real", "texto_procesado"]
if list(datos.columns) != esperadas:
    raise ValueError("Ejecuta primero el cuaderno 05 actualizado.")

candidatos = datos.loc[~datos["id"].isin(muestra["id"]), esperadas].copy()
print("Textos por clasificar:", len(candidatos), "| muestra excluida:", len(muestra))
display(candidatos.head(10))


## C. Tokenizar y obtener probabilidades

- **Qué hace:** aplica el preprocesador de RoBERTuito, divide cada texto en tokens y procesa bloques de hasta 128 posiciones.
- **Para qué:** convertir el texto en números que la red neuronal pueda recibir y evitar perder el final de publicaciones largas.
- **Por qué:** RoBERTuito fue entrenado con su propio vocabulario; usar otro tokenizador rompería la correspondencia entre tokens y embeddings.

**Tokenización no es embedding:** la tokenización produce identificadores enteros; la capa de embeddings del modelo transforma luego esos identificadores en vectores. `softmax` convierte los tres puntajes finales en valores que suman 1.


In [ ]:
MODELO = "pysentimiento/robertuito-sentiment-analysis"
DISPOSITIVO = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(MODELO)
modelo = AutoModelForSequenceClassification.from_pretrained(MODELO).to(DISPOSITIVO).eval()

mapa = {0: "negativa", 1: "neutral", 2: "positiva"}
if {int(k): v for k, v in modelo.config.id2label.items()} != {0: "NEG", 1: "NEU", 2: "POS"}:
    raise ValueError("El orden de clases del modelo cambió.")

def predecir(texto):
    entrada = preprocess_tweet(str(texto), lang="es")
    lote = tokenizer(
        entrada,
        max_length=128,
        truncation=True,
        stride=32,
        return_overflowing_tokens=True,
        padding=True,
        return_tensors="pt",
    )
    lote.pop("overflow_to_sample_mapping", None)
    lote = {nombre: tensor.to(DISPOSITIVO) for nombre, tensor in lote.items()}
    with torch.inference_mode():
        logits = modelo(**lote).logits
        probabilidades = torch.softmax(logits, dim=-1).mean(dim=0)
    return mapa[int(probabilidades.argmax())]

ejemplo = candidatos.iloc[0]["texto_procesado"]
tokens_ejemplo = tokenizer.tokenize(preprocess_tweet(ejemplo, lang="es"))[:30]
print("Dispositivo:", DISPOSITIVO)
print("Ejemplo de tokens:", tokens_ejemplo)


## D. Generar las propuestas

- **Qué hace:** aplica la función anterior a cada publicación y conserva solo la clase ganadora.
- **Para qué:** crear una hoja sencilla que pueda revisarse manualmente.
- **Por qué:** las probabilidades y el número de fragmentos son cálculos técnicos; no se necesitan como columnas del corpus.


In [ ]:
sugerencias = []
for numero, texto in enumerate(candidatos["texto_procesado"], start=1):
    sugerencias.append(predecir(texto))
    if numero % 100 == 0:
        print(f"Procesados: {numero}/{len(candidatos)}")

propuestas = candidatos.copy()
propuestas["polaridad_sugerida"] = sugerencias
propuestas["etiqueta_humana"] = ""
propuestas["observacion_humana"] = ""

print("Distribución de sugerencias:", propuestas["polaridad_sugerida"].value_counts().to_dict())
display(propuestas.head(10))


## E. Guardar la hoja de revisión

- **Qué hace:** exporta siete columnas y un resumen metodológico separado.
- **Para qué:** permitir revisar la propuesta sin mezclarla con la muestra independiente.
- **Por qué:** una etiqueta sugerida por IA y una etiqueta confirmada por una persona representan evidencias distintas.


In [ ]:
CARPETA = RAIZ / "TABLAS/06_clasificacion"
CARPETA.mkdir(parents=True, exist_ok=True)
SALIDA = CARPETA / "clasificacion_asistida_v4.xlsx"
RESUMEN = CARPETA / "resumen_clasificacion_v4.json"

if SALIDA.exists():
    anterior = pd.read_excel(SALIDA, sheet_name="clasificacion", keep_default_na=False)
    if set(anterior["id"]) != set(propuestas["id"]):
        raise ValueError("El archivo existente corresponde a otra versión del corpus.")
    revision = anterior.set_index("id")[["etiqueta_humana", "observacion_humana"]]
    propuestas = propuestas.set_index("id")
    propuestas[["etiqueta_humana", "observacion_humana"]] = revision
    propuestas = propuestas.reset_index()

with pd.ExcelWriter(SALIDA, engine="openpyxl") as escritor:
    propuestas.to_excel(escritor, sheet_name="clasificacion", index=False)
    hoja = escritor.sheets["clasificacion"]
    hoja.freeze_panes = "A2"
    hoja.auto_filter.ref = hoja.dimensions
    hoja.column_dimensions["C"].width = 65
    hoja.column_dimensions["D"].width = 65
    validacion = DataValidation(type="list", formula1='"positiva,negativa,neutral"', allow_blank=True)
    hoja.add_data_validation(validacion)
    col = get_column_letter(propuestas.columns.get_loc("etiqueta_humana") + 1)
    validacion.add(f"{col}2:{col}{hoja.max_row}")

resumen = {
    "version": "v4",
    "modelo": MODELO,
    "dispositivo": str(DISPOSITIVO),
    "max_tokens": 128,
    "solapamiento": 32,
    "regla_texto_largo": "promedio de softmax de todos los bloques",
    "filas_clasificadas": int(len(propuestas)),
    "muestra_excluida": int(len(muestra)),
    "distribucion_sugerida": {str(k): int(v) for k, v in propuestas["polaridad_sugerida"].value_counts().items()},
    "advertencia": "polaridad_sugerida es automática; etiqueta_humana requiere revisión.",
}
RESUMEN.write_text(json.dumps(resumen, ensure_ascii=False, indent=2), encoding="utf-8")
print("Guardado:", SALIDA)
print("Columnas finales:", list(propuestas.columns))
